In [ ]:
import mne
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt
from mne.decoding import CSP
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_val_score, StratifiedKFold

In [ ]:
file_path = "D:/Users/windows/Desktop/Inner Speech Decoding/Dataset/sub-01_ses-01_eeg-epo.fif"
epochs = mne.read_epochs(file_path, preload=True)
print(epochs)
label_map = {31: 'up', 32: 'down', 33: 'right', 34: 'left'}
classes = sorted(label_map.keys())
sfreq = epochs.info['sfreq']
data = epochs.get_data(copy=True)       # (n_epochs, n_channels, n_times)
labels = epochs.events[:, 2]
n_epochs, n_channels, n_times = data.shape

In [ ]:
# Filter bank: 30 bands, 0-60 Hz, 2 Hz intervals
band_edges = [(max(1, f), f + 2) for f in range(0, 60, 2)]

def bandpass(signal, lo, hi, fs, order=4):
    nyq = fs / 2.0
    lo_n = max(lo / nyq, 1e-4)
    hi_n = min(hi / nyq, 1 - 1e-4)
    b, a = butter(order, [lo_n, hi_n], btype='band')
    return filtfilt(b, a, signal, axis=-1)

In [ ]:
# Band power features: (n_epochs, n_channels, n_bands)
n_bands = len(band_edges)
features = np.zeros((n_epochs, n_channels, n_bands))
for b_idx, (lo, hi) in enumerate(band_edges):
    filtered = bandpass(data, lo, hi, sfreq)
    features[:, :, b_idx] = (filtered ** 2).mean(axis=-1)

print(f"Feature matrix shape: {features.shape}")

In [ ]:
# Fisher Ratio per class vs. rest
fisher_ratios = np.zeros((len(classes), n_channels, n_bands))
for c_idx, code in enumerate(classes):
    mask = labels == code
    X_c, X_r = features[mask], features[~mask]
    mu_c, mu_r = X_c.mean(0), X_r.mean(0)
    var_c, var_r = X_c.var(0), X_r.var(0)
    fisher_ratios[c_idx] = (mu_c - mu_r) ** 2 / (var_c + var_r + 1e-12)

mean_fr = fisher_ratios.mean(axis=0)   # (n_channels, n_bands)

In [ ]:
# Select top-K (channel, band) pairs by Fisher Ratio
top_k = 20
flat_idx = np.argsort(mean_fr.ravel())[::-1][:top_k]
ch_top, band_top = np.unravel_index(flat_idx, mean_fr.shape)
print(f"Top {top_k} Fisher Ratio features:")
for ci, bi in zip(ch_top[:5], band_top[:5]):
    lo, hi = band_edges[bi]
    print(f"  Ch {epochs.ch_names[ci]}  {lo:.0f}\u2013{hi:.0f} Hz  FR={mean_fr[ci,bi]:.4f}")

In [ ]:
# Use selected features as input to CSP + SVM pipeline
# Build 3-D array for CSP: (n_epochs, n_selected_channels, n_times) for the most discriminative band
best_band_idx = band_top[0]
lo_best, hi_best = band_edges[best_band_idx]
print(f"Best band: {lo_best:.0f}\u2013{hi_best:.0f} Hz")

data_filtered = bandpass(data, lo_best, hi_best, sfreq)  # (n_epochs, n_ch, n_times)

# CSP + SVM: binary classification (up vs down as example)
mask_binary = np.isin(labels, [31, 32])
X_bin = data_filtered[mask_binary]
y_bin = labels[mask_binary]

csp = CSP(n_components=4, reg='ledoit_wolf', log=True, norm_trace=False)
clf = make_pipeline(csp, StandardScaler(), SVC(kernel='linear', C=1.0))

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(clf, X_bin, y_bin, cv=cv, scoring='balanced_accuracy')
print(f"CSP+SVM balanced accuracy (up vs down, {lo_best:.0f}\u2013{hi_best:.0f} Hz): "
      f"{scores.mean():.3f} \u00b1 {scores.std():.3f}")

In [ ]:
# Fisher Ratio heatmap for visual comparison
band_centers = [(lo + hi) / 2 for lo, hi in band_edges]
fig, ax = plt.subplots(figsize=(14, 5))
im = ax.imshow(mean_fr, aspect='auto', origin='lower',
               extent=[band_centers[0], band_centers[-1], 0, n_channels])
ax.set_xlabel('Band center frequency (Hz)')
ax.set_ylabel('Channel index')
ax.set_title('Filter Bank + Fisher Ratio Heatmap')
plt.colorbar(im, ax=ax, label='Fisher Ratio')
plt.tight_layout()
plt.show()